In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pickle

from sf_workingdir_lilly.dmlab.custom_weight_generator import return_weights_for_iso_feat
from sf_workingdir_lilly.dmlab.analysis.custom_esn_analysis import get_hidden_states_iso_feat


# after Memory of Recurrent Networks: Do We Compute It Right? (Giovanni et al., 2024)

In [3]:
# after Memory of Recurrent Networks: Do We Compute It Right? (Giovanni et al., 2024)
# Kryglov Conditioning

In [4]:
# idea: do a PCA on the Kryglov matrix of the network
# if columns are linearly dependent, then the network has low memory capacity, because older inputs do not create new independent state directions

In [5]:
spec_rad = 1.4
time_steps = 70

Hippo_n_feature = 16
Hippo_R = 8
Hippo_L = 64
trial=0
fixed_whh=True
fixed_wih=True

x_vals = np.arange(0,70)
weight_trials = [1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 22, 23, 24, 27, 28, 29, 30, 33, 35, 36, 38, 39, 40, 41, 42, 43, 45, 47, 49]

In [6]:
# 1) construct the Krylov matrix
def construct_krylov_matrix(spec_rad, time_steps, trial=trial):
    # C: w_ih, A: w_hh
    C, A = return_weights_for_iso_feat(spec_rad, trial=trial, fixed_whh=True, fixed_wih=True, vary_sparsity=False, sparsity=0.2, folder='weights')
    K = torch.zeros((C.shape[0], time_steps))
    # input 
    u = torch.zeros(Hippo_n_feature)
    u[0] = 1.0
    # create a varaibel, so that the power of A does not need to be computed each time
    current_A = torch.linalg.matrix_power(A, 0)
    # C @ input
    C = C @ u
    for t in range(time_steps):
        col = current_A @ C 
        current_A = current_A @ A
        K[:, t] = col  
    return K

In [7]:
K = construct_krylov_matrix(spec_rad, time_steps, trial=trial)
print("Krylov matrix shape:", K.shape)

Krylov matrix shape: torch.Size([1136, 70])


In [8]:
# do PCA on K by doing SVD and then using W_m

K = construct_krylov_matrix(spec_rad, time_steps, trial=trial) 

# W_m: projection matrix in lag space (its columns: orthonormal basis of the lag directions that survive in the state), tells us which delays are still represented
U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)
print(Vh.shape, Vh)

# P_m = W_m @ W_m.T: projection matrix in lag space
P = Vh.T @ Vh

print(P.shape, P)

MC = torch.diag(P)

N = K.shape[0]
m = K.shape[1]
print(N, m)
print('rank of K:', torch.linalg.matrix_rank(K).item())


/tmp/ipykernel_779808/3568107484.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)


torch.Size([70, 70]) tensor([[ 8.1453e-09,  2.3842e-07,  5.8208e-11,  ..., -3.5705e-01,
         -4.9989e-01, -6.9987e-01],
        [-1.2258e-05,  2.3132e-05,  2.0742e-05,  ...,  5.6620e-01,
          7.7481e-02, -5.1239e-01],
        [-2.6719e-06,  1.1582e-05,  2.7768e-05,  ...,  1.9612e-01,
          5.0024e-01, -1.3845e-01],
        ...,
        [ 8.9393e-01, -1.4991e-01,  1.6677e-01,  ..., -1.2190e-02,
          3.8064e-03,  1.6160e-03],
        [ 0.0000e+00, -2.2181e-01, -3.4541e-01,  ...,  2.9369e-02,
          1.6663e-02, -3.8401e-02],
        [ 4.3628e-01,  4.6961e-01, -3.1904e-01,  ...,  2.4353e-02,
         -1.2081e-02,  1.1176e-04]])
torch.Size([70, 70]) tensor([[ 1.0000e+00, -4.4329e-08, -2.8641e-08,  ..., -2.7940e-09,
         -1.6298e-08, -2.3974e-09],
        [-4.4329e-08,  1.0000e+00,  1.0298e-08,  ..., -5.5879e-09,
         -1.7462e-07, -1.3709e-07],
        [-2.8641e-08,  1.0298e-08,  1.0000e+00,  ...,  9.8255e-08,
          5.6112e-08, -1.2669e-07],
        ...,
    

In [10]:

# TODO double check, but only project onto the row space of the vectors that correspond to the nonzero singular values
tol = S.max() * max(K.shape) * torch.finfo(S.dtype).eps
k = int((S > tol).sum().item())
print("effective k:", k)  

# threshold the singular values
Vh_k = Vh[:k, :]           # (k, m)
P = Vh_k.T @ Vh_k          # (m, m) projector onto row-space
print(P.shape)
MC = torch.diag(P)         # (m,)
print(MC.shape)
print(MC.min().item(), MC.max().item(), MC.sum().item())
print(MC)

effective k: 3
torch.Size([70, 70])
torch.Size([70])
9.638689586061178e-13 0.7715262770652771 3.0000011920928955
tensor([1.5740e-10, 6.6929e-10, 1.2013e-09, 2.8301e-10, 3.6394e-12, 9.6387e-13,
        6.5855e-12, 7.8500e-12, 1.2193e-11, 2.0694e-11, 4.7795e-11, 1.0636e-10,
        6.5025e-11, 1.3393e-10, 3.5746e-10, 2.6472e-10, 4.5096e-10, 7.6095e-10,
        1.2961e-09, 2.8093e-09, 3.0125e-09, 4.8992e-09, 1.0636e-08, 1.2486e-08,
        1.4630e-08, 2.9776e-08, 4.5039e-08, 4.9808e-08, 9.1239e-08, 1.3058e-07,
        1.7178e-07, 3.4798e-07, 3.5681e-07, 6.6149e-07, 1.3562e-06, 1.0934e-06,
        2.2075e-06, 4.0097e-06, 4.0040e-06, 7.5131e-06, 9.5193e-06, 1.2005e-05,
        2.7351e-05, 3.2890e-05, 3.9125e-05, 8.5212e-05, 1.0620e-04, 1.3386e-04,
        2.8351e-04, 3.5360e-04, 4.8482e-04, 8.2975e-04, 7.8684e-04, 1.6470e-03,
        3.2978e-03, 2.9928e-03, 5.1015e-03, 7.4542e-03, 8.1386e-03, 2.3750e-02,
        3.5738e-02, 3.7860e-02, 5.5668e-02, 5.0352e-02, 1.2503e-01, 3.7803e-01,
       

In [9]:
print(MC)
print(MC.sum())

tensor([1.5740e-10, 6.6929e-10, 1.2013e-09, 2.8301e-10, 3.6394e-12, 9.6387e-13,
        6.5855e-12, 7.8500e-12, 1.2193e-11, 2.0694e-11, 4.7795e-11, 1.0636e-10,
        6.5025e-11, 1.3393e-10, 3.5746e-10, 2.6472e-10, 4.5096e-10, 7.6095e-10,
        1.2961e-09, 2.8093e-09, 3.0125e-09, 4.8992e-09, 1.0636e-08, 1.2486e-08,
        1.4630e-08, 2.9776e-08, 4.5039e-08, 4.9808e-08, 9.1239e-08, 1.3058e-07,
        1.7178e-07, 3.4798e-07, 3.5681e-07, 6.6149e-07, 1.3562e-06, 1.0934e-06,
        2.2075e-06, 4.0097e-06, 4.0040e-06, 7.5131e-06, 9.5193e-06, 1.2005e-05,
        2.7351e-05, 3.2890e-05, 3.9125e-05, 8.5212e-05, 1.0620e-04, 1.3386e-04,
        2.8351e-04, 3.5360e-04, 4.8482e-04, 8.2975e-04, 7.8684e-04, 1.6470e-03,
        3.2978e-03, 2.9928e-03, 5.1015e-03, 7.4542e-03, 8.1386e-03, 2.3750e-02,
        3.5738e-02, 3.7860e-02, 5.5668e-02, 5.0352e-02, 1.2503e-01, 3.7803e-01,
        4.9754e-01, 4.8653e-01, 5.0614e-01, 7.7153e-01])
tensor(3.0000)


In [10]:
def compute_mc(spec_rad, time_steps, trial=trial):
    K = construct_krylov_matrix(spec_rad, time_steps, trial=trial) 
    U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)

    tol = S.max() * max(K.shape) * torch.finfo(S.dtype).eps
    k = int((S > tol).sum().item())
    #print("effective k:", k)  

    # threshold the singular values
    Vh_k = Vh[:k, :]           # (k, m)
    P = Vh_k.T @ Vh_k          # (m, m) projector onto row-space
    MC = torch.diag(P)         # (m,) 
    return MC

In [11]:
MC_mode = compute_mc(spec_rad, time_steps, trial=trial)
print(MC_mode)
print(MC_mode.sum())

tensor([1.5740e-10, 6.6929e-10, 1.2013e-09, 2.8301e-10, 3.6394e-12, 9.6387e-13,
        6.5855e-12, 7.8500e-12, 1.2193e-11, 2.0694e-11, 4.7795e-11, 1.0636e-10,
        6.5025e-11, 1.3393e-10, 3.5746e-10, 2.6472e-10, 4.5096e-10, 7.6095e-10,
        1.2961e-09, 2.8093e-09, 3.0125e-09, 4.8992e-09, 1.0636e-08, 1.2486e-08,
        1.4630e-08, 2.9776e-08, 4.5039e-08, 4.9808e-08, 9.1239e-08, 1.3058e-07,
        1.7178e-07, 3.4798e-07, 3.5681e-07, 6.6149e-07, 1.3562e-06, 1.0934e-06,
        2.2075e-06, 4.0097e-06, 4.0040e-06, 7.5131e-06, 9.5193e-06, 1.2005e-05,
        2.7351e-05, 3.2890e-05, 3.9125e-05, 8.5212e-05, 1.0620e-04, 1.3386e-04,
        2.8351e-04, 3.5360e-04, 4.8482e-04, 8.2975e-04, 7.8684e-04, 1.6470e-03,
        3.2978e-03, 2.9928e-03, 5.1015e-03, 7.4542e-03, 8.1386e-03, 2.3750e-02,
        3.5738e-02, 3.7860e-02, 5.5668e-02, 5.0352e-02, 1.2503e-01, 3.7803e-01,
        4.9754e-01, 4.8653e-01, 5.0614e-01, 7.7153e-01])
tensor(3.0000)


/tmp/ipykernel_2491879/4144140653.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)


In [12]:
summed_MC = []
#x_vals = np.arange(0,70)
#fig, ax = plt.subplots(10,5, figsize=(25,50), sharex=True)
for t in range(50):
    #for i in range(Hippo_n_feature):
    current_MC = compute_mc(spec_rad, time_steps, trial=t)
    summed_MC.append(current_MC.sum().item())
    #ax[t//5, t%5].plot(current_MC)
    #ax[t//5, t%5].set_title(f'{t}')
print("Summed MC over trials:", summed_MC)

/tmp/ipykernel_2491879/4144140653.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)


Summed MC over trials: [3.0000011920928955, 7.0, 6.000001430511475, 10.000000953674316, 14.999999046325684, 6.999999046325684, 7.000001907348633, 6.000000953674316, 4.000001430511475, 5.000000953674316, 6.000000476837158, 7.0, 9.000001907348633, 7.0, 6.000001430511475, 10.999999046325684, 6.999999523162842, 5.000000953674316, 12.99999713897705, 11.000000953674316, 8.999999046325684, 14.000001907348633, 9.999998092651367, 9.999998092651367, 5.999999523162842, 3.0, 11.000001907348633, 7.000000953674316, 12.00000286102295, 7.000000953674316, 5.000000953674316, 8.999999046325684, 9.000000953674316, 5.999998092651367, 7.000000476837158, 6.999999523162842, 12.000000953674316, 11.999998092651367, 7.999999523162842, 4.999999046325684, 6.000000953674316, 3.999999523162842, 10.0, 8.0, 3.999997615814209, 4.999997615814209, 8.999999046325684, 13.000000953674316, 7.000000953674316, 9.0]


In [13]:
print(len(summed_MC))

50


In [14]:
# TODO include sanity check: sum(MC)≈rank(K)

# Xiaos' method SVD - singular value threshold

In [15]:
# idea: sum the singular values: that is the max variance, then look how many singular values are needed to reach 90% (or other threshold) of that variance

In [16]:
def compute_mc_svd_variance(spec_rad, time_steps, trial=trial, threshold=0.98):
    K = construct_krylov_matrix(spec_rad, time_steps, trial=trial) 
    U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)

    total_variance = (S**2).sum().item()
    # why squared singular values? because variance along each principal component is given by the square of the singular value
    variance_cumsum = torch.cumsum(S**2, dim=0)

    num_singular_values = 1
    i = 0
    while variance_cumsum[i] <= threshold * total_variance:
        num_singular_values += 1
        i += 1

    return num_singular_values

In [17]:
MC_singular_components = []
for t in range(50):
    current_MC = compute_mc_svd_variance(spec_rad, time_steps, trial=t)
    MC_singular_components.append(current_MC)

print("MC singular components over trials:", MC_singular_components)

/tmp/ipykernel_2491879/810337113.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  U, S, Vh = torch.linalg.svd(torch.tensor(K), full_matrices=False)


MC singular components over trials: [1, 3, 2, 4, 4, 2, 2, 1, 1, 1, 1, 2, 3, 1, 3, 2, 4, 1, 3, 2, 1, 4, 1, 2, 2, 1, 2, 2, 2, 4, 1, 2, 1, 2, 2, 2, 4, 3, 2, 1, 2, 2, 2, 2, 1, 1, 2, 2, 3, 3]


In [18]:
print(len(MC_singular_components))

50


# singular value threshold but with the hidden states, so the nonlinearity is included

In [19]:
def compute_svd_variance_hs(spec_rad, time_steps, trial=trial, threshold=0.98):
    hs = get_hidden_states_iso_feat(time_steps, spec_rad, Hippo_n_feature, Hippo_R, Hippo_L, trial, fixed_whh, fixed_wih, input_idx=0)  # (time_steps, hidden_size)
    U, S, Vh = torch.linalg.svd(torch.tensor(hs.T), full_matrices=False)

    total_variance = (S**2).sum().item()
    variance_cumsum = torch.cumsum(S**2, dim=0)
    num_singular_values = 1
    i = 0
    while variance_cumsum[i] <= threshold * total_variance:
        num_singular_values += 1
        i += 1

    return num_singular_values

In [20]:
MC_singular_components_hs = []
for t in range(50):
    current_hs_MC = compute_svd_variance_hs(spec_rad, time_steps, trial=t)
    MC_singular_components_hs.append(current_hs_MC)

print("MC singular components from hidden states over trials:", MC_singular_components_hs)


/tmp/ipykernel_2491879/2687054017.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  U, S, Vh = torch.linalg.svd(torch.tensor(hs.T), full_matrices=False)


MC singular components from hidden states over trials: [4, 8, 7, 6, 9, 8, 7, 7, 7, 7, 1, 6, 9, 7, 6, 10, 1, 7, 7, 1, 2, 1, 6, 6, 6, 1, 1, 7, 8, 6, 6, 1, 1, 10, 1, 9, 8, 1, 10, 5, 7, 7, 7, 5, 4, 7, 6, 8, 1, 9]


# Participation ratio

In [21]:
def compute_participation_ratio(spec_rad, time_steps, trial=trial, threshold=0.98):
    K = construct_krylov_matrix(spec_rad, time_steps, trial=trial) 
    K = K.to(dtype=torch.float64) 
    U, S, Vh = torch.linalg.svd(K, full_matrices=False)
    #S/=1e5
    eig_vals = S**2
    #print("eigenvalues:", eig_vals)
    pr = (eig_vals.sum().item())**2 / (eig_vals**2).sum().item()
    return pr

current_pr = compute_participation_ratio(spec_rad, time_steps, trial=22)
print("Participation ratio:", current_pr)

Participation ratio: 1.008298094529003


In [22]:
def compute_participation_ratio_norm(spec_rad, time_steps, trial=trial, threshold=0.98):
    K = construct_krylov_matrix(spec_rad, time_steps, trial=trial) 
    U, S, Vh = torch.linalg.svd(K, full_matrices=False)
    S/=1e5
    eig_vals = S**2
    #print("eigenvalues:", eig_vals)
    pr = (eig_vals.sum().item())**2 / (eig_vals**2).sum().item()
    return pr

current_pr2 = compute_participation_ratio_norm(spec_rad, time_steps, trial=22)
print("Participation ratio norm:", current_pr2)

Participation ratio norm: 1.008297882636862


In [23]:
participation_ratios = []
participation_ratios2 = []
for t in range(50):
    current_pr = compute_participation_ratio(spec_rad, time_steps, trial=t)
    participation_ratios.append(current_pr)
    current_pr2 = compute_participation_ratio_norm(spec_rad, time_steps, trial=t)
    participation_ratios2.append(current_pr2)

print("Participation ratios over trials:", participation_ratios)
print("Participation ratios norm over trials:", participation_ratios2)

Participation ratios over trials: [1.0000013219404338, 2.1037921153299055, 1.605778320249779, 2.2790331907853707, 1.9000551238095709, 1.7996419134157788, 1.6216218378546172, 1.0340032243668296, 1.0000559347080722, 1.0002652966333818, 1.0016451729717366, 1.8818978036607263, 1.9088256148622065, 1.001865467306432, 2.058103421268259, 1.9426994305604366, 1.9685495300749742, 1.0004074421946063, 1.3949111695228507, 1.3628852847945232, 1.008252229692457, 1.3991509900117123, 1.008298094529003, 1.8832614946046609, 1.7773350646041466, 1.0000117884679531, 1.160466151449945, 1.7227466794813695, 1.5013328126843903, 2.1174693248025735, 1.0000015363162678, 1.7611655560337949, 1.0006163785270166, 1.3934947309300105, 1.0545064419243322, 1.2002397955051478, 1.9109272104524517, 1.9675484715803753, 1.7270035657971647, 1.0000006310679317, 1.46498953950203, 1.7441421271125341, 1.7779904828096618, 1.7531579956469179, 1.0000010233721666, 1.0216487616476526, 1.2786559957664656, 1.160528397240699, 1.675319259669

# add to the MC measures dictionary

In [24]:
with open("/home/fr/fr_lr554/samplefactory/sample-factory/sf_workingdir_lilly/dmlab/analysis/data/mc_measures1.4.pkl", "rb") as f:
    mc_measures = pickle.load(f)

In [25]:
summed_MC_stable = [summed_MC[i] for i in weight_trials]
MC_singular_components_stable = [MC_singular_components[i] for i in weight_trials]
MC_singular_components_hs_stable = [MC_singular_components_hs[i] for i in weight_trials]
participation_ratios_stable = [participation_ratios[i] for i in weight_trials]


In [26]:
mc_measures['summed_MC_svd'] = summed_MC_stable
mc_measures['MC_singular_components_svd'] = MC_singular_components_stable
mc_measures['MC_singular_components_hs_svd'] = MC_singular_components_hs_stable
mc_measures['participation_ratio_svd'] = participation_ratios_stable

In [27]:
with open("/home/fr/fr_lr554/samplefactory/sample-factory/sf_workingdir_lilly/dmlab/analysis/data/mc_measures1.4.pkl", "wb") as f:
    pickle.dump(mc_measures, f)